In [ ]:
import socket

HOST = '0.0.0.0'  # Listen on all interfaces
PORT = 12345       # Port to listen on

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind((HOST, PORT))
    s.listen()
    print(f"Listening on {HOST}:{PORT}")
    while True:
      conn, addr = s.accept()
      with conn:
          print(f"Connected by {addr}")
          while True:
              data = conn.recv(1024)
              if not data:
                  break
              print(data.decode())

Listening on 0.0.0.0:12345
Connected by ('192.168.1.113', 52027)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52028)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52029)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52030)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52031)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52032)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52033)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52034)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52035)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52036)
Time: 31s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52037)
Time: 32s | Message: Hello from ESP32-CAM!
Connected by ('192.168.1.113', 52038)
Time: 32s | Message: Hello from ESP32-CAM!
C

In [ ]:
model1 = YOLO(r"C:\Users\pc\OneDrive\Desktop\AI\weights\yolo12n_combined.pt").cuda()
#video = cv2.VideoCapture(0)
path =r'C:\Users\pc\OneDrive\Desktop\AI\images\gfddfds.jpg'
frame = cv2.imread(path)
results = model1.predict(path,conf=0.5)
for result in results:
    for result in results:
        boxes =result.boxes.xyxy.cpu().numpy()
        classes =result.boxes.cls.cpu().numpy()
        confidences = result.boxes.conf.cpu().numpy()
        for box, cls,conf in zip(boxes,classes,confidences):
                
                x1, y1, x2, y2 = map(int, box[:4])
                # print(str(int(conf*100))+"%")
                cv2.rectangle(frame, (x1, y1), (x2, y2), (8, 255, 0), 2)
                cv2.putText(frame, result.names[int(cls)] +" "+ str(int(conf*100))+"%", (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
plt.imshow(frame)
plt.show()

In [ ]:
import socket
import cv2
import numpy as np
import time
import numpy as np
from ultralytics import YOLO
import cv2
import matplotlib.pyplot as plt
import math

# TCP server configuration
HOST = '0.0.0.0'  # Listen on all interfaces
PORT = 12346      # Port to listen on
model1 = YOLO(r"C:\Users\pc\OneDrive\Desktop\AI\weights\yolov12n_traffic2.pt").cuda()
model2= YOLO(r"C:\Users\pc\OneDrive\Desktop\AI\weights\yolo11n.pt").cuda()
# Create a TCP server
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.bind((HOST, PORT))
    s.listen()
    print(f"Listening on {HOST}:{PORT}")
    frame_count = 0
    start_time = time.time()
    while True:
     conn, addr = s.accept()
     with conn:
         # print(f"Connected by {addr}")
         while True:
             # Receive the frame size
             frame_size_data = conn.recv(4)
             if not frame_size_data:
                 break
             frame_size = int.from_bytes(frame_size_data, byteorder='little')
    
             # Receive the frame data
             frame_data = b''
             while len(frame_data) < frame_size:
                 packet = conn.recv(frame_size - len(frame_data))
                 if not packet:
                     break
                 frame_data += packet
    
             # Convert the frame data to an image
             frame = cv2.imdecode(np.frombuffer(frame_data, dtype=np.uint8), cv2.IMREAD_COLOR)
             if frame is not None:
                #  frame=cv2.cvtColor(frame,cv2.COLOR_RGB2BGR)
                 results1 = model1.predict(frame,conf=0.5)
                 results2 = model2.predict(frame,conf=0.5)
                 for result in results1:
                     for result in results1:
                         boxes =result.boxes.xyxy.cpu().numpy()
                         classes =result.boxes.cls.cpu().numpy()
                         confidences = result.boxes.conf.cpu().numpy()
                         for box, cls,conf in zip(boxes,classes,confidences):
                             x1, y1, x2, y2 = map(int, box[:4])
                            # #  y3=y2+(y1-y2)//2
                            #  # print(str(int(conf*100))+"%")
                             cv2.line(frame, (x1+(x2-x1)//2,y1), (x1+(x2-x1)//2,y2), (0, 200, 0), 3)
                             w = math.hypot(x2 - x1, 0)
                            #  # # print(w)
                             W = 10
                        
                            #  # # # # Finding the Focal Length
                            #  d = 50
                            #  f = (w*d)/W
                            #  print(f)
                        
                            #  # # # Finding distance
                             f = 320
                             d = (W * f) / w
                             print(d)
                        
                             
                            #  # print(str(int(conf*100))+"%")
                             cv2.rectangle(frame, (x1, y1), (x2, y2), (8, 255, 0), 2)
                             cv2.putText(frame, result.names[int(cls)] +" "+ str(int(conf*100))+"%", (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
                             cv2.putText(frame,f'{int(d)}cm', (x1, y1+25), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
                             if d<53:
                                cv2.putText(frame, "stop car", (x1, y1+50), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)
                 for result in results2:
                  
                    boxes =result.boxes.xyxy.cpu().numpy()
                    classes =result.boxes.cls.cpu().numpy()
                    confidences = result.boxes.conf.cpu().numpy()
                    for box, cls ,conf in zip(boxes,classes,confidences):
                        if result.names[int(cls)] == 'car':
                            x1, y1, x2, y2 = map(int, box[:4])
                            cv2.rectangle(frame, (x1, y1), (x2, y2), (8, 255, 0), 2)
                            cv2.putText(frame, result.names[int(cls)] +" "+ str(int(conf*100))+"%", (x1, y1), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0,0,255), 2)

                #  frame=cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
                 # Display the frame
                 cv2.imshow('ESP32-CAM Stream', frame)
    
                #  # Save the frame to disk
                #  frame_filename = r"C:\Users\pc\OneDrive\Desktop\AI\tests2\frame_"+str(frame_count)+".jpg"
                #  cv2.imwrite(frame_filename, frame)
                #  print(f"Frame saved as {frame_filename}")
                #  frame_count += 1
                 
    
             # Exit on 'q' key press
     if cv2.waitKey(1) & 0xFF == ord('q'):
       print("kdfgsh")
       print("kdfgsh")
       endtime = time.time()
       break

                 
cv2.destroyAllWindows()



Listening on 0.0.0.0:12345


In [2]:
import socket
import keyboard

# Replace with the ESP32-CAM's IP address
ESP32_IP = "192.168.1.106"
PORT = 12345

# Create a TCP client
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((ESP32_IP, PORT))
    print(f"Connected to {ESP32_IP}:{PORT}")

    # Listen for keypresses
    print("Press any key to send (or 'q' to quit)...")
    while True:
        try:
            # Wait for a keypress event
            event = keyboard.read_event()
            if event.event_type == keyboard.KEY_DOWN:  # Only send on keydown
                key = event.name  # Get the key that was pressed
                if key == 'q':  # Exit on 'q' key
                    print("Exiting...")
                    break
                # Send the key to the ESP32-CAM
                s.sendall(key.encode())
                print(f"Sent '{key}' to ESP32-CAM")
        except Exception as e:
            print(f"Error: {e}")
            break
#hi aaaaaaaaassssssaaaaaaaaaaaaaaaaaaqqqqqaaaaaaaaaaaaaaqaaaasdfghasdfadsadassadsdasdasdsaqmmmasasdfghfdsddsdsadddfgfdsasasas

KeyboardInterrupt: 

In [6]:
import socket

# Replace with the ESP32-CAM's IP address
ESP32_IP = "192.168.1.106"
PORT = 12345

# Create a TCP client
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
    s.connect((ESP32_IP, PORT))
    print(f"Connected to {ESP32_IP}:{PORT}")

    while True:
        # Wait for user input
        user_input = input("Press 'q' to quit: ")

        s.sendall(user_input.encode()+b'\n')
        print(f"Sent ' {user_input} 'to ESP32-CAM")
        if user_input == 'q':
            # Exit the loop
            break

Connected to 192.168.1.106:12345


Sent ' 25 'to ESP32-CAM
Sent ' gfdg 'to ESP32-CAM
Sent ' fdg 'to ESP32-CAM
Sent ' dfg 'to ESP32-CAM
Sent ' dfgfdgd 'to ESP32-CAM
Sent ' dfgfdgdf 'to ESP32-CAM
Sent ' dfgdfg 'to ESP32-CAM
Sent ' fgfgdfg 'to ESP32-CAM
Sent ' dfg 'to ESP32-CAM
Sent ' fdg 'to ESP32-CAM
Sent ' dfg 'to ESP32-CAM
Sent ' fff 'to ESP32-CAM
Sent ' 35 'to ESP32-CAM
Sent ' 34 'to ESP32-CAM
Sent ' 34 'to ESP32-CAM
Sent ' 34 'to ESP32-CAM
Sent ' 34 'to ESP32-CAM
Sent ' 45 'to ESP32-CAM
Sent ' 20 'to ESP32-CAM
Sent ' 60 'to ESP32-CAM
Sent ' 50 'to ESP32-CAM
Sent ' 70 'to ESP32-CAM
Sent ' 30 'to ESP32-CAM
Sent ' 60 'to ESP32-CAM
Sent ' 60 'to ESP32-CAM
Sent ' 70 'to ESP32-CAM
Sent ' 80 'to ESP32-CAM
Sent ' 100 'to ESP32-CAM
Sent ' 105 'to ESP32-CAM
Sent ' 110 'to ESP32-CAM
Sent ' 70 'to ESP32-CAM
Sent ' 20 'to ESP32-CAM
Sent ' 25 'to ESP32-CAM
Sent ' 30 'to ESP32-CAM
Sent ' 40 'to ESP32-CAM
Sent ' 50 'to ESP32-CAM
Sent ' 60 'to ESP32-CAM
Sent ' 30 'to ESP32-CAM
Sent ' 60 'to ESP32-CAM
Sent ' 25 'to ESP32-CAM
Sent ' 60

In [16]:
frame_count/(endtime-start_time)

16.253882393021385